In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone

PROTOCOL_ID = "TRKH_PRETRAINED_CLASSF_B0_20260731"
RUN_TAG = "kaggle_b0_v1"
AUTO_RESUME = False
CONFIRM_FULL = False  # Chi doi True sau khi doc metrics probe.
RUN_PROBE = not AUTO_RESUME
PROBE_MIN_MACRO_F1 = 0.55
PROBE_MIN_CLASS1_F1 = 0.30
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
DATA_YAML_SHA256 = "312eb376e89023f42e9df24fea8723b64a6b885c15c24f85d8e319d1ff4f1fa8"
IMAGE_TREE_SHA256 = "70a1b7d2b4c6f80e28fe3f0f714f1ba3e8ab654a90ce50b1e9cae8e4dac4a503"
EXPECTED_SOURCE_COMMIT = "73c96f3d8f42e80c62ab2c4e3c0691ff81f45b77"
EXPECTED_SOURCE_TREE_SHA256 = "d7de5ed0dcd5d9c4eb6e0eeaaef2939505749a6831c7b0885cde172ec822003a"
DINO_SHA256 = "2a1ec16ae28ffa07bc0ead0241ee7df9fc26451fe6f9f839b7b3afa0a906b040"
DINO_BYTES = 86362376
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print(PROTOCOL_ID, "auto_resume=", AUTO_RESUME, "run_probe=", RUN_PROBE)


In [ ]:
# Cai dung version; khi tat Internet, dinh kem wheelhouse trong /kaggle/input.
from importlib.metadata import PackageNotFoundError, version

def ensure_locked_packages(requirements):
    pending = []
    for distribution, (module, expected) in requirements.items():
        try:
            observed = version(distribution)
        except PackageNotFoundError:
            observed = None
        if observed != expected or importlib.util.find_spec(module) is None:
            pending.append(f"{distribution}=={expected}")
    if pending:
        wheel_dirs = sorted({str(path.parent) for path in INPUT_ROOT.rglob("*.whl")})
        installed = False
        if wheel_dirs:
            command = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-index"]
            for wheel_dir in wheel_dirs:
                command += ["--find-links", wheel_dir]
            installed = subprocess.run(command + pending, check=False).returncode == 0
        if not installed:
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *pending], check=True)
    for distribution, (_, expected) in requirements.items():
        if version(distribution) != expected:
            raise RuntimeError(f"Version lock failed: {distribution}={version(distribution)} expected={expected}")

ensure_locked_packages({
    "timm": ("timm", "1.0.27"),
    "safetensors": ("safetensors", "0.7.0"),
    "PyYAML": ("yaml", "6.0.3"),
    "Pillow": ("PIL", "11.3.0"),
    "matplotlib": ("matplotlib", "3.9.4"),
    "pytest": ("pytest", "8.4.2"),
})
print("Core dependencies ready")


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

repo_roots = sorted({path.parents[2] for path in INPUT_ROOT.rglob("trkh/recipes/pretrained_classf_b0.py")})
if len(repo_roots) != 1:
    raise RuntimeError(f"Can dung 1 source TRKH_pretrained duy nhat, tim thay: {repo_roots}")
REPO_ROOT = repo_roots[0]

data_candidates = [path for path in INPUT_ROOT.rglob("data.yaml") if sha256(path) == DATA_YAML_SHA256]
if len(data_candidates) != 1:
    raise RuntimeError(f"Can dung 1 class_f/data.yaml dung SHA-256, tim thay: {data_candidates}")
DATA_YAML = data_candidates[0]
DATA_ROOT = DATA_YAML.parent

weight_candidates = []
for path in INPUT_ROOT.rglob("*.safetensors"):
    if path.is_file() and path.stat().st_size == DINO_BYTES and sha256(path) == DINO_SHA256:
        weight_candidates.append(path)
if len(weight_candidates) != 1:
    raise RuntimeError(f"Can dung 1 DINOv3 weight dung hash/kich thuoc, tim thay: {weight_candidates}")
DINO_WEIGHT = weight_candidates[0]

sys.path.insert(0, str(REPO_ROOT))
os.environ["PYTHONPATH"] = str(REPO_ROOT)
os.environ["MPLBACKEND"] = "Agg"
os.chdir(REPO_ROOT)
if re.fullmatch(r"[0-9a-f]{40}", EXPECTED_SOURCE_COMMIT) is None:
    raise RuntimeError("Notebook release chua duoc khoa source commit")
if re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SOURCE_TREE_SHA256) is None:
    raise RuntimeError("Notebook release chua duoc khoa source-tree digest")
source_digest = hashlib.sha256()
source_files = sorted((REPO_ROOT / "trkh").rglob("*.py")) + sorted((REPO_ROOT / "configs").rglob("*.yaml"))
for path in source_files:
    source_digest.update(path.relative_to(REPO_ROOT).as_posix().encode("utf-8") + b"\0")
    source_digest.update(path.read_bytes().replace(b"\r\n", b"\n"))
observed_source_tree_sha256 = source_digest.hexdigest()
if observed_source_tree_sha256 != EXPECTED_SOURCE_TREE_SHA256:
    raise RuntimeError(f"Sai source snapshot: {observed_source_tree_sha256} != {EXPECTED_SOURCE_TREE_SHA256}")
SOURCE_COMMIT = EXPECTED_SOURCE_COMMIT
SOURCE_TREE_SHA256 = EXPECTED_SOURCE_TREE_SHA256
print("repo =", REPO_ROOT)
print("source_commit =", SOURCE_COMMIT, "source_tree_sha256 =", SOURCE_TREE_SHA256)
print("data =", DATA_YAML)
print("dino =", DINO_WEIGHT)


In [ ]:
import yaml
from trkh.core.config import load_data_spec
from trkh.recipes.pretrained_classf_b0 import validate_development_data_yaml
from trkh.tools.validate_canonical_classf import validate_canonical_classf

document = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
document["path"] = str(DATA_ROOT)
document.pop("test", None)
DEV_YAML = WORK_ROOT / "class_f_dev_test_locked.yaml"
DEV_YAML.write_text(yaml.safe_dump(document, sort_keys=False, allow_unicode=True), encoding="utf-8")
dataset_contract = validate_canonical_classf(DATA_YAML)
assert dataset_contract["image_tree_sha256"] == IMAGE_TREE_SHA256
CLASSF_ATTESTATION = WORK_ROOT / "class_f_canonical_attestation.json"
CLASSF_ATTESTATION.write_text(json.dumps(dataset_contract, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
development_contract = validate_development_data_yaml(DEV_YAML, canonical_contract=dataset_contract)
dev_spec = load_data_spec(DEV_YAML, class_name_mode="raw", expected_num_classes=5)
assert not dev_spec.has_test_split, "Test lock failed"
assert list(dev_spec.class_names) == list(dataset_contract["class_names"])
print("Static canonical contract passed; test is absent from all development commands:", DEV_YAML)


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Hay bat GPU Accelerator trong Kaggle Settings")
properties = torch.cuda.get_device_properties(0)
vram_gib = properties.total_memory / (1024 ** 3)
if vram_gib >= 14.5:
    MICRO_BATCH = 24
elif vram_gib >= 9.5:
    MICRO_BATCH = 16
else:
    MICRO_BATCH = 12
GRAD_ACCUM = {24: 2, 16: 3, 12: 4, 8: 6}[MICRO_BATCH]
AMP_DTYPE = "bf16" if properties.major >= 8 else "fp16"
EVAL_BATCH = 64 if vram_gib >= 14.5 else 32
os.environ["TRKH_AMP_DTYPE"] = AMP_DTYPE
os.environ.setdefault("OMP_NUM_THREADS", "4")

PROBE_TAG = f"{RUN_TAG}_probe"
FULL_TAG = f"{RUN_TAG}_full"

def recipe_command(mode, run_tag, auto_resume=False):
    command = [
        sys.executable, "-m", "trkh.recipes.pretrained_classf_b0",
        "--mode", mode,
        "--data", DATA_YAML,
        "--canonical-attestation", CLASSF_ATTESTATION,
        "--train-data", DEV_YAML,
        "--dino-checkpoint", DINO_WEIGHT,
        "--output-dir", RUNS_ROOT,
        "--run-tag", run_tag,
        "--batch-size", MICRO_BATCH,
        "--num-workers", "4",
        "--eval-num-workers", "2",
        "--seed", "42",
        "--amp-dtype", AMP_DTYPE,
        "--source-commit", SOURCE_COMMIT,
        "--source-tree-sha256", SOURCE_TREE_SHA256,
    ]
    if mode == "full":
        command.append("--confirm-full")
    if auto_resume:
        command.append("--auto-resume")
    return [str(item) for item in command]

print(properties.name, f"{vram_gib:.1f} GiB", AMP_DTYPE, f"batch={MICRO_BATCH}x{GRAD_ACCUM}")
print("probe:", shlex.join(recipe_command("probe", PROBE_TAG)))
print("full:", shlex.join(recipe_command("full", FULL_TAG, AUTO_RESUME)))


In [ ]:
def run_checked(arguments, required=True):
    command = [str(item) for item in arguments]
    print("\n$", shlex.join(command), flush=True)
    result = subprocess.run(command, cwd=REPO_ROOT, env=os.environ.copy(), check=False)
    if required and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)
    return result.returncode

PROBE_DIR = RUNS_ROOT / f"pretrained_dinov3_classf_direct_probe_{PROBE_TAG}"
if RUN_PROBE:
    probe_marker_path = PROBE_DIR / "b0_stage_complete.json"
    if not probe_marker_path.is_file():
        run_checked(recipe_command("probe", PROBE_TAG))
    probe_marker = json.loads(probe_marker_path.read_text(encoding="utf-8"))
    probe_metrics = probe_marker.get("metrics", {})
    if probe_marker.get("returncode") != 0 or float(probe_metrics.get("best_macro_f1") or 0) < PROBE_MIN_MACRO_F1:
        raise RuntimeError(f"Probe gate failed: {probe_marker}")
    if float(probe_metrics.get("best_class1_f1") or 0) < PROBE_MIN_CLASS1_F1:
        raise RuntimeError(f"Probe did not learn canonical class 1: {probe_metrics}")
    print("Probe gate passed:", probe_metrics)
    if not CONFIRM_FULL:
        raise RuntimeError("Da dung sau probe. Hay xem metrics, doi CONFIRM_FULL=True va chay lai cell nay de mo full train.")

run_checked(recipe_command("full", FULL_TAG, AUTO_RESUME))
RUN_DIR = RUNS_ROOT / f"pretrained_dinov3_classf_direct_full_{FULL_TAG}"
FULL_PREFLIGHT_MANIFEST = RUNS_ROOT / f"preflight_classf_b0_{FULL_TAG}" / "full_manifest.json"
BEST_CHECKPOINT = RUN_DIR / "checkpoints" / "best.pt"
if not BEST_CHECKPOINT.is_file() or not FULL_PREFLIGHT_MANIFEST.is_file():
    raise FileNotFoundError(f"Missing best/preflight: {BEST_CHECKPOINT}, {FULL_PREFLIGHT_MANIFEST}")
print("Best checkpoint:", BEST_CHECKPOINT)


In [ ]:
EVAL_DIR = RUN_DIR / "eval_val_full"
run_checked([
    sys.executable, "-m", "trkh.evaluation.evaluate",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--split", "val",
    "--batch-size", EVAL_BATCH,
    "--num-workers", "2",
    "--amp",
    "--max-batches", "0",
    "--paper-name", "TRKH-DINOv3-ClassF-B0",
    "--family", "TRKH-Pretrained",
    "--output-dir", EVAL_DIR,
])
PREDICTIONS = EVAL_DIR / "predictions_detailed.csv"
if not PREDICTIONS.is_file():
    PREDICTIONS = EVAL_DIR / "predictions.csv"
if not PREDICTIONS.is_file():
    raise FileNotFoundError("Evaluate did not write predictions CSV")
METRICS_DETAIL = EVAL_DIR / "metrics_detailed.json"
if not METRICS_DETAIL.is_file():
    METRICS_DETAIL = EVAL_DIR / "metrics.json"
print(METRICS_DETAIL.read_text(encoding="utf-8")[:4000])


In [ ]:
run_checked([
    sys.executable, "-m", "trkh.tools.audit_class_confusions",
    "--predictions", PREDICTIONS,
    "--data", DEV_YAML,
    "--focus-class-index", "1",
    "--top-k-images", "32",
    "--copy-images",
    "--output-dir", RUN_DIR / "audit_class1_confusions",
])
run_checked([
    sys.executable, "-m", "trkh.tools.audit_prediction_forensics",
    "--predictions", PREDICTIONS,
    "--pairs", "0-1,1-2,1-4,2-3",
    "--focus-class-index", "1",
    "--ece-bins", "15",
    "--image-stats-mode", "foreground",
    "--max-image-stats", "512",
    "--image-stats-workers", "2",
    "--top-k-images", "32",
    "--copy-images",
    "--output-dir", RUN_DIR / "audit_prediction_forensics",
])


In [ ]:
run_checked([
    sys.executable, "-m", "trkh.evaluation.xai_audit",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--split", "val",
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--output-dir", RUN_DIR / "xai_val",
    "--max-cases", "24",
    "--mistake-cases", "8",
    "--low-confidence-cases", "4",
    "--close-margin-cases", "4",
    "--per-class-cases", "1",
    "--focus-class-index", "1",
    "--focus-false-positive-cases", "4",
    "--focus-false-negative-cases", "4",
    "--batch-size", EVAL_BATCH,
    "--num-workers", "2",
    "--method", "gradcam",
    "--feature-source", "patch_embed",
    "--robustness-probes",
])
run_checked([
    sys.executable, "-m", "trkh.evaluation.robustness_eval",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--batch-size", EVAL_BATCH,
    "--num-workers", "2",
    "--max-batches", "0",
    "--num-fail-cases", "16",
    "--output-dir", RUN_DIR / "robustness_val",
])


In [ ]:
run_checked([
    sys.executable, "-m", "trkh.tools.trace_architecture",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--image-size", "256",
    "--device", "cuda",
    "--seed", "42",
    "--output-dir", RUN_DIR / "architecture_trace",
])

# DINOv3 B0 la teacher/control, khong phai mobile model. Day chi la technical export.
deploy_code = -1
try:
    ensure_locked_packages({"onnx": ("onnx", "1.19.1")})
except Exception as error:
    print("WARNING: bo qua ONNX export vi dependency offline:", repr(error))
else:
    deploy_code = run_checked([
        sys.executable, "-m", "trkh.inference.deploy",
        "--checkpoint", BEST_CHECKPOINT,
        "--data", DEV_YAML,
        "--class-name-mode", "raw",
        "--expected-num-classes", "5",
        "--output-dir", RUN_DIR / "deploy_teacher_fp32",
        "--batch-size", EVAL_BATCH,
        "--num-workers", "2",
        "--benchmark-batch-size", "1",
        "--benchmark-warmup", "2",
        "--benchmark-runs", "2",
        "--split", "val",
        "--skip-accuracy",
        "--skip-benchmark",
        "--skip-trt-engine",
    ], required=False)
    if deploy_code:
        print("WARNING: technical ONNX export failed; checkpoint/audits remain valid. returncode=", deploy_code)


In [ ]:
full_preflight = json.loads(FULL_PREFLIGHT_MANIFEST.read_text(encoding="utf-8"))
validation_metrics = json.loads(METRICS_DETAIL.read_text(encoding="utf-8"))
validation_support = sum(int(row["support"]) for row in validation_metrics["per_class"])
with PREDICTIONS.open("r", encoding="utf-8-sig", newline="") as handle:
    prediction_rows = sum(1 for _ in csv.DictReader(handle))
if validation_support != 2479 or prediction_rows != 2479:
    raise RuntimeError(f"Full validation incomplete: support={validation_support}, predictions={prediction_rows}")
class1_metrics = validation_metrics["per_class"][1]
audit_artifacts = {
    "validation": METRICS_DETAIL,
    "predictions": PREDICTIONS,
    "class1_confusions": RUN_DIR / "audit_class1_confusions",
    "prediction_forensics": RUN_DIR / "audit_prediction_forensics",
    "xai": RUN_DIR / "xai_val" / "xai_audit_summary.json",
    "robustness": RUN_DIR / "robustness_val" / "robustness_summary.json",
    "architecture_trace": RUN_DIR / "architecture_trace" / "trace_summary.json",
}
audit_status = {
    name: (path.is_file() or (path.is_dir() and any(path.rglob("*"))))
    for name, path in audit_artifacts.items()
}
if not all(audit_status.values()):
    raise RuntimeError(f"Required audit missing: {audit_status}")
shutil.copy2(DEV_YAML, RUN_DIR / DEV_YAML.name)
shutil.copy2(CLASSF_ATTESTATION, RUN_DIR / CLASSF_ATTESTATION.name)
shutil.copy2(FULL_PREFLIGHT_MANIFEST, RUN_DIR / "b0_full_preflight_manifest.json")
if (PROBE_DIR / "b0_stage_complete.json").is_file():
    shutil.copy2(PROBE_DIR / "b0_stage_complete.json", RUN_DIR / "b0_probe_stage_complete.json")
evidence_roots = list(audit_artifacts.values()) + [
    BEST_CHECKPOINT,
    RUN_DIR / DEV_YAML.name,
    RUN_DIR / CLASSF_ATTESTATION.name,
    RUN_DIR / "b0_full_preflight_manifest.json",
]
evidence_files = set()
for root in evidence_roots:
    if root.is_file():
        evidence_files.add(root)
    elif root.is_dir():
        evidence_files.update(path for path in root.rglob("*") if path.is_file())
evidence_inventory = [
    {
        "path": path.relative_to(RUN_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    for path in sorted(evidence_files)
]
EVIDENCE_INVENTORY = RUN_DIR / "evidence_sha256_inventory.json"
EVIDENCE_INVENTORY.write_text(json.dumps(evidence_inventory, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
runtime_distributions = {}
for distribution in ("torch", "torchvision", "timm", "safetensors", "numpy", "Pillow", "PyYAML", "matplotlib", "pytest", "onnx", "onnxruntime"):
    try:
        runtime_distributions[distribution] = version(distribution)
    except PackageNotFoundError:
        runtime_distributions[distribution] = None
manifest = {
    "schema_version": 1,
    "protocol": PROTOCOL_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_data_yaml_sha256": DATA_YAML_SHA256,
    "canonical_image_tree_sha256": IMAGE_TREE_SHA256,
    "canonical_attestation_sha256": sha256(CLASSF_ATTESTATION),
    "source_commit": SOURCE_COMMIT,
    "source_tree_sha256": SOURCE_TREE_SHA256,
    "dino_sha256": DINO_SHA256,
    "best_checkpoint": str(BEST_CHECKPOINT),
    "best_checkpoint_sha256": sha256(BEST_CHECKPOINT),
    "validation_only": True,
    "validation_support": validation_support,
    "validation_prediction_rows": prediction_rows,
    "validation_metrics": {
        "accuracy": validation_metrics["accuracy"],
        "macro_precision": validation_metrics["macro_precision"],
        "macro_recall": validation_metrics["macro_recall"],
        "macro_f1": validation_metrics["macro_f1"],
        "class1": class1_metrics,
    },
    "audit_status": audit_status,
    "evidence_inventory": str(EVIDENCE_INVENTORY),
    "evidence_inventory_sha256": sha256(EVIDENCE_INVENTORY),
    "evidence_inventory_entries": len(evidence_inventory),
    "test_static_contract_verified": True,
    "test_metadata_and_bytes_used_for_static_integrity_audit": True,
    "test_pixels_used_by_model": False,
    "test_metrics_or_model_selection_used": False,
    "test_metrics_read": False,
    "old_dataset_checkpoint_or_teacher_used": False,
    "development_data": development_contract,
    "full_preflight_manifest_sha256": sha256(FULL_PREFLIGHT_MANIFEST),
    "train_args": full_preflight["train_args"],
    "python": sys.version,
    "runtime_distributions": runtime_distributions,
    "gpu": properties.name,
    "vram_gib": vram_gib,
    "amp_dtype": AMP_DTYPE,
    "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM,
    "deploy_returncode": deploy_code,
}
MANIFEST = RUN_DIR / "kaggle_artifact_manifest.json"
MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
archive_base = WORK_ROOT / f"TRKH_CLASSF_B0_{RUN_TAG}_RESULTS"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Download:", archive_path)
